<a href="https://colab.research.google.com/github/anbaluxe/my_colab_learning/blob/main/CUPED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_users = 1000

df = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'group': np.random.choice(
        ['Control', 'Treatment'],
        size=n_users
    )
})

# Метрика пользователя ДО эксперимента
df['X'] = np.random.gamma(
    shape=2,
    scale=500,
    size=n_users
)

# Метрика пользователя ВО ВРЕМЯ эксперимента
# Она частично зависит от X
df['Y'] = (
    100
    + 0.3 * df['X']
    + np.random.normal(0, 250, n_users)
)

# Небольшой эффект эксперимента
df.loc[df['group'] == 'Treatment', 'Y'] += 50

df.head()

,user_id,group,X,Y
0,1,Control,1073.974630,426.979333
1,2,Treatment,2725.798408,946.517068
2,3,Control,377.532084,-97.571357
3,4,Control,1192.904048,217.499122
4,5,Control,238.619972,379.299073


Подсчитаем Корреляцию

In [2]:
df[['X', 'Y']].corr()

,X,Y
X,1.000000,0.656518
Y,0.656518,1.000000


Высчитывает `ϴ`

In [3]:
cov_xy = df['X'].cov(df['Y'])
var_x = df['X'].var()

teta = cov_xy / var_x

print(teta)

0.30839962254797615


Посчитаем Y cuped

In [21]:
df['Y_cuped'] = df['Y'] - teta * df['X']

df[['Y', 'Y_cuped']].var()

,0
Y,105397.323555
Y_cuped,59969.453030


Проверим сам A/B эффект

In [10]:
mean_y = df.groupby('group')['Y'].mean()
mean_cuped = df.groupby('group')['Y_cuped'].mean()

delta_y = mean_y['Treatment'] - mean_y['Control']
delta_y_cuped = mean_cuped['Treatment'] - mean_cuped['Control']

print('Эффект Y:', delta_y)
print('Эффект Y_cuped:', delta_y_cuped)

Эффект Y: 48.683934710628535
Эффект Y_cuped: 41.95507954758713


In [6]:
mean_x = df.groupby('group')['X'].mean()

print(mean_x)
print(mean_y)

group
Control      1012.157043
Treatment    1034.003975
Name: X, dtype: float64
group
Control      402.035685
Treatment    450.719620
Name: Y, dtype: float64


In [16]:
var_y = df.groupby('group')['Y'].var()
var_y_cuped = df.groupby('group')['Y_cuped'].var()
n = df.groupby('group').size()

In [17]:
se_y = (
    var_y['Treatment']/n['Treatment']
    + var_y['Control']/n['Control']) ** 0.5
se_y_cuped = (
    var_y_cuped['Treatment']/n['Treatment']
    + var_y_cuped['Control']/n['Control']) ** 0.5

print(se_y)
print(se_y_cuped)

20.458591472577524
15.438246619907334


In [18]:
ci_left_y = delta_y - 1.96 * se_y
ci_right_y = delta_y + 1.96 * se_y
ci_y = (ci_left_y, ci_right_y)

In [19]:
ci_left_y_cuped = delta_y_cuped - 1.96 * se_y_cuped
ci_right_y_cuped = delta_y_cuped + 1.96 * se_y_cuped
ci_y_cuped = (ci_left_y_cuped, ci_right_y_cuped)

In [20]:
print(ci_y)
print(ci_y_cuped)

(np.float64(8.58509542437659), np.float64(88.78277399688048))
(np.float64(11.696116172568757), np.float64(72.2140429226055))
